In [19]:
import random as rd
import pygraphviz as pgv
import networkx as nx
from networkx.drawing.nx_agraph import from_agraph
import json
import datetime

print(datetime.datetime.now())
map_data = pgv.AGraph("map_data.dot")
entanglements = pgv.AGraph("entanglements.dot")
with open('units.json') as ujson:
    units = json.load(ujson)
with open('orders.txt') as otxt:
    orders_l = [line.rstrip() for line in otxt]

### Order syntax = unit:type:origin:destination:status
# unit is a handle composed of a number from 1 to 6 (each associated with one of the powers in play) and a letter which uniquely identifies the unit
# type is the type of order, expressed as the order's initial : H to Hold, T to Tunnel, S to Support, M to Measure, C to Convoy
# origin is a three-letter handle for the province of origin of that specific order. For all orders, it is the province that the unit occupies (in the case of Convoying, it is identical to the start province of the
    # corresponding tunnel order
# destination is also a three-letter handle for the targeted province. For Tunnels (and Convoys related to them), as well as supports, it is a province adjacent to origin. For Holds and Measures, origin and destination are
    # the same
# status is always 0, unless the order is a tunneling order that is being convoyed (in which case it should be 1).

### For support and convoy orders, they are followed on the same line by the order they are supporting/convoying, in the form = unit:type:origin:destination:status:unit:type:origin:destination:status
### The first line of an order.txt file should have the name of the current turn capitalized (SPRING, FALL or WINTER)

def dict_creation(journal,element):
    journal['unit'] = element[0]
    journal['type'] = element[1]
    journal['origin'] = element[2]
    journal['destination'] = element[3]
    journal['convoyed'] = bool(int(element[4]))
    return journal

def province_calc(provinces,units):
    provinces = {value: 0 for value in provinces}
    for unit in units.keys():
        for n in units[unit]['superposition']:
            provinces[n] += 100 // len(units[unit]['superposition'])
    return provinces

if orders_l[0] == "SPRING" or orders_l[0] == "FALL":
    #Manufacturing order dictionary from shorthand list
    orders = []
    for item in orders_l[1:]:
        element = item.split(":")
        journal = {}
        target = {}
        if len(element) == 5:
            journal = dict_creation(journal,element)
            journal['score'] = 0
            orders.append(journal)
        elif len(element) == 10:
            target = dict_creation(target,element[5:])
            target['score'] = 0
            journal = dict_creation(journal,element[:5])
            journal['score'] = 0
            journal['target'] = target
            orders.append(journal)
        else:
            print("Error - invalid order syntax : " + item)

    def compat(utype,tgtype):
        return tgtype == 'coast' or (tgtype == 'land' and utype == 'Army') or (tgtype == 'sea' and utype == 'Fleet')

    def coastline(origin,destination):
        value = False
        for tile in map_data.neighbors(order['origin']):
            if tile in map_data.neighbors(order['destination']) and map_data.get_node(tile).attr['type'] == 'sea':
                value = True
        return value

    def adj(origin,destination):
        return map_data.has_neighbor(origin,destination)

    def supincl(unit,origin):
        return origin in unit['superposition']

    #Handling order contradictions
    orders = [order for order in orders.copy() if order['unit'] in units.keys()]
    order_unit_pair = {}
    invalid_keys = set()
    for order in orders:
        if order['unit'] not in order_unit_pair:
            order_unit_pair[order['unit']] = order['type']
        elif order_unit_pair[order['unit']] != order['type']:
            invalid_keys.add(order['unit'])
            print("Contradicting orders assigned to " + order['unit'])
    orders = [order for order in orders.copy() if order['unit'] not in invalid_keys]

    #Handling convoys
    for order in orders.copy():
        if order['type'] == 'C':
            if order['target'] not in orders or not order['target']['convoyed']:
                print("Convoy by " + order['unit'] + " invalid : no matching Tunnel order for " + order['target']['unit'])
                orders.remove(order)
            elif not supincl(order['unit'],order['origin']) or not supincl(order['unit'],order['destination']):
                print("Invalid convoy by " + order['unit'] + " from " + order['origin'] + " to " + order['destination'])
                orders[orders.index(order['target'])]['convoyed'] == False
                orders.remove(order)
            else:
                entanglements.add_edge(order['target']['unit'],order['unit'])

    #Handling repeat measurements
    for order in orders.copy():
        if order['type'] == 'M' and units[order['unit']]['mflag'] == True:
            print("Cannot measure " + order['unit'] + " two consecutive turns")
            orders.remove(order)
    for unit in units.keys():
        units[unit]['mflag'] = False

    #Handling supports (targetting)
    for order in orders.copy():
        if order['type'] == 'S':
            if order['target'] not in orders:
                print("Support by " + order['unit'] + " invalid : no matching order for " + order['target']['unit'])
                orders.remove(order)
            elif not (order['target']['type'] == 'T' or order['target']['type'] == 'H' or order['target']['type'] == 'M'):
                print("Invalid order to be supported by " + order['unit'])
                orders.remove(order)

    #Handling province contradictions
    for order in orders.copy():
        if supincl(units[order['unit']],order['origin']):
            if not order['convoyed'] and (order['type'] == 'T' or order['type'] == 'S'):
                if not adj(order['origin'],order['destination']):
                    print(order['unit'] + " cannot influence " + order['destination'] + " from " + order['origin'] + " (not adjacent)")
                    orders.remove(order)
                elif not compat(units[order['unit']]['type'],map_data.get_node(order['destination']).attr['type']):
                    print(orders['unit'] + " cannot influence " + order['destination'] + " from " + order['origin'] + " (tile incompatible)")
                    orders.remove(order)
                elif map_data.get_node(order['destination']).attr['type'] == 'coast' and map_data.get_node(order['origin']).attr['type'] == 'coast' and units[order['unit']]['type'] == 'Fleet' and not coastline(order['origin'],order['destination']):
                    print(order['unit'] + " cannot influence " + order['destination'] + " from " + order['origin'] + " (no shared coastline)")
                    orders.remove(order)
        else:
            print(order['origin'] + " is not part of " + order['unit'] + "'s superposition.")
            orders.remove(order)

    #Handling cut supports and attributing support values to their corresponding order + support-induced entanglements
    atkd_provinces = []
    for order in orders:
        if order['type'] == 'T' or order['type'] == 'M':
            atkd_provinces.append(order['destination'])
    for order in orders.copy():
        if order['type'] == 'S' and (order['origin'] not in atkd_provinces):
            sup_value = units[order['unit']]['superposition'].count(order['origin']) * (100 // len(units[order['unit']]['superposition']))
            orders[orders.index(order['target'])]['score'] += sup_value
            entanglements.add_edge(order['target']['unit'],order['unit'])
        elif order['type'] == 'S' and (order['origin'] in atkd_provinces):
            print(str(order['unit']) + "'s support from " + str(order['origin']) + " was cut")
            orders.remove(order)

    #Handling holds
    for order in orders.copy():
        if order['type'] == 'H':
            for attack in orders:
                if (attack['type'] == 'T' or attack['type'] == 'M') and order['destination'] == attack['destination']:
                    attack['score'] -= order['score'] + units[order['unit']]['superposition'].count(order['destination']) * (100 // len(units[order['unit']]['superposition']))

    #Handling tunnel and measuring success
    prov_units = units.copy()
    success_keys = []
    failed_keys = []
    for order in orders:
        if order['type'] == 'T' or order['type'] == 'M':
            order['score'] += units[order['unit']]['superposition'].count(order['origin']) * (100 // len(units[order['unit']]['superposition']))
            roll = rd.randint(1,100)
            result = order['score'] - roll
            if result < 0:
                print(str(order['unit']) + " " + str(order['type']) + " from " + str(order['origin']) + " to " + str(order['destination']) + " attempt failed with success rate = " + str(order['score']) + " from roll = " + str(roll))
                failed_keys.append(order)
            else:
                print(str(order['unit']) + " " + str(order['type']) + " from " + str(order['origin']) + " to " + str(order['destination']) +  " attempt succeeded with success rate = " + str(order['score']) + " from roll = " + str(roll))
                success_keys.append(order)
    for order in success_keys:
        if order['type'] == 'T':
            prov_units[order['unit']]['superposition'].append(order['destination'])
        if order['type'] == 'M':
            prov_units[order['unit']]['superposition'] = [order['destination']]
            prov_units[order['unit']]['mflag'] = True
            if order['unit'] in entanglements.nodes():
                subset = nx.node_connected_component(from_agraph(entanglements),order['unit'])
                subset = list(filter((order['unit']).__ne__, subset))
                print(str(subset) + " need to be measured as a result of entanglement with " + order['unit'])
                entanglements.remove_nodes_from(subset)
    for order in failed_keys:
        if order['type'] == 'M' : prov_units[order['unit']]['superposition'] = list(filter((order['destination']).__ne__, prov_units[order['unit']]['superposition']))

    #Handling bounces and retreats
    provinces = map_data.nodes()
    provinces = {value: 0 for value in provinces}
    
    def bounce(orders,provinces,nunits,ounits):
        for i, order in enumerate(orders):
            for match in orders[i+1:]:
                if provinces[order['destination']] > 100 and (order['destination'] == match['destination']) and order['type'] == 'T' and order['unit'] != match['unit']:
                    print(order['unit'] + " and " + match['unit'] + " bounced in " + order['destination'])
                    nunits[order['unit']]['superposition'] = list(filter((order['destination']).__ne__, nunits[order['unit']]['superposition']))
                    if match['type'] == 'M':
                        nunits[match['unit']]['superposition'] = ounits[match['unit']]['superposition']
                    else:
                        nunits[match['unit']]['superposition'] = list(filter((match['destination']).__ne__, nunits[match['unit']]['superposition']))
                    provinces = province_calc(provinces,nunits)
                    orders, provinces, nunits = bounce(orders,provinces,nunits,ounits)
                elif order['type'] == 'M' and match['type'] == 'M' and (order['destination'] == match['destination']):
                    print(order['unit'] + " and " + match['unit'] + " bounced in " + order['destination'])
                    nunits[order['unit']]['superposition'] = ounits[order['unit']]['superposition']
                    nunits[match['unit']]['superposition'] = ounits[match['unit']]['superposition']
        return orders, provinces, nunits

    provinces = province_calc(provinces,prov_units)
    success_keys, provinces, prov_units = bounce(success_keys, provinces, prov_units, units)
    
    for order in success_keys:
        if provinces[order['destination']] > 100:
            for unit in prov_units.keys():
                if order['destination'] in prov_units[unit]['superposition'] and unit != order['unit']:
                    print(unit + " was dislodged from " + order['destination'])
                    prov_units[unit]['superposition'] = list(filter((order['destination']).__ne__, prov_units[unit]['superposition']))
                    for n in prov_units[unit]['superposition']:
                        if (provinces[n] - (100 // (1+len(prov_units[unit]['superposition']))) + (100 // len(prov_units[unit]['superposition']))) > 100 :
                            prov_units[unit]['superposition'] = list(filter((n).__ne__, prov_units[unit]['superposition']))
                            provinces = province_calc(provinces,prov_units)
    for unit in prov_units.copy().keys():
        if not prov_units[unit]['superposition']:
            print(unit + "was disbanded for lack of a valid retreat")
            if unit in entanglements.nodes():
                subset = nx.node_connected_component(from_agraph(entanglements),unit)
                subset = list(filter((unit).__ne__, subset))
                for i in subset:
                    print(i + "was disbanded due to entanglement with" + unit)
                    del prov_units[n]
                entanglements.remove_nodes_from(subset)
            del prov_units[unit]

    reserve = 100*len(prov_units.keys())
    provinces = province_calc(provinces,prov_units)
    for key in provinces:
        reserve -= provinces[key]
    print("Reserve at " + str(reserve) + "% capacity")
    print("Province percentages : ", provinces)

elif orders_l[0] == "WINTER":
    prov_units = units.copy()
    provinces = map_data.nodes()
    provinces = {value: 0 for value in provinces}
    provinces = province_calc(provinces,prov_units)
    
    #Handling build and retreats
    for item in orders_l[1:]:
        element = item.split(":")
        prov_units[element[0]] = {'superposition':[element[1]],'type':element[2],'mflag':False}
        if provinces[element[1]] > 0:
            for unit in prov_units:
                if element[1] in prov_units[unit]['superposition']:
                    print(unit + " was dislodged from " + element[1])
                    prov_units[unit]['superposition'] = list(filter((element[1]).__ne__, prov_units[unit]['superposition']))
                    for n in prov_units[unit]['superposition']:
                        if (provinces[n] - (100 // (1+len(prov_units[unit]['superposition']))) + (100 // len(prov_units[unit]['superposition']))) > 100 :
                            prov_units[unit]['superposition'] = list(filter((n).__ne__, prov_units[unit]['superposition']))
                            provinces = province_calc(provinces,prov_units)
        provinces = province_calc(provinces,prov_units)
    
    for unit in prov_units.copy().keys():
        if not prov_units[unit]['superposition']:
            print(unit + "was disbanded for lack of a valid retreat")
            if unit in entanglements.nodes():
                subset = nx.node_connected_component(from_agraph(entanglements),unit)
                subset = list(filter((unit).__ne__, subset))
                for i in subset:
                    print(i + "was disbanded due to entanglement with" + unit)
                    del prov_units[n]
                entanglements.remove_nodes_from(subset)
            del prov_units[unit]

2026-07-31 19:59:04.254810
4A M from Rhi to Rhi attempt failed with success rate = 33 from roll = 41
4B M from War to War attempt succeeded with success rate = 50 from roll = 16
4E M from Hun to Hun attempt succeeded with success rate = 100 from roll = 2
5B M from Lit to Lit attempt succeeded with success rate = 100 from roll = 85
5E M from BOT to BOT attempt failed with success rate = 50 from roll = 100
Reserve at 4% capacity
Province percentages :  {'Ice': 40, 'NAO': 60, 'NWG': 20, 'IRI': 0, 'Ire': 40, 'CEL': 50, 'MAO': 25, 'BAR': 0, 'NTH': 50, 'Nor': 20, 'Stp': 100, 'Hol': 0, 'Sct': 20, 'Yor': 50, 'Lon': 50, 'ENG': 25, 'Bel': 75, 'Swe': 100, 'SKA': 0, 'Fin': 0, 'BOT': 0, 'Est': 0, 'Lat': 100, 'Mos': 0, 'Den': 0, 'BAL': 0, 'HEL': 0, 'Min': 0, 'Lit': 100, 'Vol': 100, 'Kiv': 0, 'Bre': 0, 'War': 100, 'Pom': 0, 'Ter': 0, 'Sil': 50, 'Cze': 0, 'Ber': 100, 'Bav': 50, 'Rhi': 0, 'Swi': 50, 'Bur': 50, 'Wal': 75, 'Bri': 25, 'Pic': 25, 'BIS': 0, 'Por': 0, 'WES': 0, 'Mor': 0, 'And': 0, 'Gas': 33,

In [20]:
with open("units.json", "w") as f:
    json.dump(prov_units, f, indent = 4)
entanglements.write("entanglements.dot")